In [93]:
import pandas as pd
import numpy as np
import os

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [95]:
# ============================================================
# FILE PATHS
# ============================================================

utilization_path = r"..\trend_prediction_model\team_handoff\trend_prediction_handoff.csv"

te_path = r"..\te_equivalence\outputs\TE_Equivalent_Alternatives.csv"


# ============================================================
# LOAD DATA
# ============================================================

utilization = pd.read_csv(utilization_path)

te_data = pd.read_csv(te_path)


print("Utilization shape:", utilization.shape)
print("TE shape:", te_data.shape)

Utilization shape: (1451, 8)
TE shape: (132, 6)


In [96]:
print("UTILIZATION COLUMNS:")
print(utilization.columns.tolist())

print("\nTE COLUMNS:")
print(te_data.columns.tolist())

UTILIZATION COLUMNS:
['Gnrc_Name', 'Brnd_Name', 'Predicted_2024_Claims', 'Predicted_2024_Benes', 'Predicted_2024_Cost', 'Cost_Growth_Pct', 'Growth_Flag', 'Note']

TE COLUMNS:
['Trade_Name', 'Ingredient', 'TE_Code', 'Alternative_Trade_Name', 'Alternative_TE_Code', 'TE_Decision']


In [97]:
# ============================================================
# CLEAN DRUG NAMES
# ============================================================

def clean_name(x):
    if pd.isna(x):
        return ""
    
    return (
        str(x)
        .upper()
        .strip()
        .replace(",", "")
        .replace("  ", " ")
    )


# Utilization names
utilization["Generic_Clean"] = (
    utilization["Gnrc_Name"]
    .apply(clean_name)
)

utilization["Brand_Clean"] = (
    utilization["Brnd_Name"]
    .apply(clean_name)
)


# TE names
te_data["Original_Clean"] = (
    te_data["Trade_Name"]
    .apply(clean_name)
)

te_data["Alternative_Clean"] = (
    te_data["Alternative_Trade_Name"]
    .apply(clean_name)
)


print("Drug names cleaned successfully.")

Drug names cleaned successfully.


In [98]:
# ============================================================
# KEEP TE-EQUIVALENT ALTERNATIVES
# ============================================================

te_pairs = te_data[
    (te_data["Original_Clean"] != "") &
    (te_data["Alternative_Clean"] != "")
].copy()


print("Total TE alternatives:", len(te_pairs))

display(
    te_pairs[
        [
            "Trade_Name",
            "Alternative_Trade_Name",
            "TE_Code",
            "Alternative_TE_Code"
        ]
    ].head(20)
)

Total TE alternatives: 132


,Trade_Name,Alternative_Trade_Name,TE_Code,Alternative_TE_Code
0,DESMOPRESSIN ACETATE (NEEDS NO REFRIGERATION),DESMOPRESSIN ACETATE,AB,AB
1,ALLOPURINOL,LOPURIN,AB,AB
2,WARFARIN SODIUM,JANTOVEN,AB,AB
3,DOXORUBICIN HYDROCHLORIDE (LIPOSOMAL),DOXORUBICIN HYDROCHLORIDE,AB,AB
4,DESMOPRESSIN ACETATE,DESMOPRESSIN ACETATE (NEEDS NO REFRIGERATION),AB,AB
5,METHENAMINE HIPPURATE,UREX,AB,AB
6,"GRISEOFULVIN, ULTRAMICROSIZE",FULVICIN P/G,AB,AB
7,DILTIAZEM HYDROCHLORIDE,CARTIA XT,AB1,AB3
8,LEVOTHYROXINE SODIUM,LEVOLET,"AB1,AB2,AB3,AB4","AB1,AB2,AB3,AB4"
9,CLARAVIS,ZENATANE,AB1,AB1


In [99]:
# ============================================================
# CREATE UTILIZATION LOOKUP
# ============================================================

generic_lookup = utilization[
    [
        "Generic_Clean",
        "Predicted_2024_Cost",
        "Predicted_2024_Benes",
        "Predicted_2024_Claims"
    ]
].copy()

generic_lookup = generic_lookup.rename(
    columns={
        "Generic_Clean": "Drug_Name",
        "Predicted_2024_Cost": "Original_Cost",
        "Predicted_2024_Benes": "Original_Benes",
        "Predicted_2024_Claims": "Original_Claims"
    }
)


brand_lookup = utilization[
    [
        "Brand_Clean",
        "Predicted_2024_Cost",
        "Predicted_2024_Benes",
        "Predicted_2024_Claims"
    ]
].copy()

brand_lookup = brand_lookup.rename(
    columns={
        "Brand_Clean": "Drug_Name",
        "Predicted_2024_Cost": "Original_Cost",
        "Predicted_2024_Benes": "Original_Benes",
        "Predicted_2024_Claims": "Original_Claims"
    }
)


# Combine generic and brand lookup
lookup = pd.concat(
    [
        generic_lookup,
        brand_lookup
    ],
    ignore_index=True
)


lookup = lookup[
    lookup["Drug_Name"] != ""
].copy()


# Remove duplicate drug names
lookup = lookup.drop_duplicates(
    subset=["Drug_Name"]
)


print("Utilization lookup created:", lookup.shape)

Utilization lookup created: (2309, 4)


In [100]:
# ============================================================
# MATCH ORIGINAL DRUG WITH UTILIZATION
# ============================================================

formulary = te_pairs.merge(
    lookup,
    left_on="Original_Clean",
    right_on="Drug_Name",
    how="left"
)


print(
    "Total TE alternatives:",
    len(formulary)
)

print(
    "Original drugs with utilization:",
    formulary["Original_Cost"].notna().sum()
)

print(
    "Original drugs without utilization:",
    formulary["Original_Cost"].isna().sum()
)

Total TE alternatives: 132
Original drugs with utilization: 41
Original drugs without utilization: 91


In [101]:
# ============================================================
# RULE-BASED ALTERNATIVE COST SIMULATION
# ============================================================

# IMPORTANT:
# Actual alternative-drug cost is not available in the
# current utilization dataset.
#
# Therefore, for the prototype simulation, we assume
# the alternative costs 15% less than the original drug.

ESTIMATED_ALTERNATIVE_REDUCTION = 15.0


formulary["Estimated_Alternative_Cost"] = np.nan


has_original_cost = (
    formulary["Original_Cost"].notna()
    &
    (formulary["Original_Cost"] > 0)
)


formulary.loc[
    has_original_cost,
    "Estimated_Alternative_Cost"
] = (
    formulary.loc[
        has_original_cost,
        "Original_Cost"
    ]
    *
    (1 - ESTIMATED_ALTERNATIVE_REDUCTION / 100)
)


print("Estimated alternative cost calculated.")

Estimated alternative cost calculated.


In [102]:
# ============================================================
# COST REDUCTION
# ============================================================

formulary["Cost_Reduction_Pct"] = np.nan


formulary.loc[
    has_original_cost,
    "Cost_Reduction_Pct"
] = (
    (
        formulary.loc[
            has_original_cost,
            "Original_Cost"
        ]
        -
        formulary.loc[
            has_original_cost,
            "Estimated_Alternative_Cost"
        ]
    )
    /
    formulary.loc[
        has_original_cost,
        "Original_Cost"
    ]
) * 100


print("Cost reduction calculated.")

Cost reduction calculated.


In [103]:
# ============================================================
# ACCESS COVERAGE
# ============================================================

# Your current utilization dataset does not contain
# actual customer satisfaction survey data or alternative
# access coverage data.
#
# Therefore, for this prototype simulation, a TE-equivalent
# alternative with available utilization is assigned
# 100% simulated access coverage.

formulary["Simulated_Access_Coverage_Pct"] = np.nan


formulary.loc[
    formulary["Original_Cost"].notna(),
    "Simulated_Access_Coverage_Pct"
] = 100.0


print("Simulated access coverage calculated.")

Simulated access coverage calculated.


In [104]:
# ============================================================
# MANDATORY FORMULARY REQUIREMENTS
# ============================================================

MIN_COST_REDUCTION = 12.0
MIN_ACCESS = 95.0


# Cost requirement
formulary["Cost_Requirement"] = (
    formulary["Cost_Reduction_Pct"]
    >= MIN_COST_REDUCTION
)


# Access requirement
formulary["Access_Requirement"] = (
    formulary["Simulated_Access_Coverage_Pct"]
    >= MIN_ACCESS
)


print("Minimum cost reduction:", MIN_COST_REDUCTION, "%")
print("Minimum access requirement:", MIN_ACCESS, "%")

Minimum cost reduction: 12.0 %
Minimum access requirement: 95.0 %


In [105]:
# ============================================================
# FINAL RULE-BASED DECISION
# ============================================================

def final_decision(row):

    # No utilization data for original drug
    if pd.isna(row["Original_Cost"]):
        return "DATA NEEDED"

    # Both mandatory requirements satisfied
    if (
        row["Cost_Requirement"]
        and
        row["Access_Requirement"]
    ):
        return "RECOMMEND SWITCH"

    # Otherwise do not switch
    return "DO NOT SWITCH"


formulary["Final_Decision"] = (
    formulary.apply(
        final_decision,
        axis=1
    )
)


print("Final decisions generated.")

Final decisions generated.


In [106]:
# ============================================================
# PMPM SAVING
# ============================================================

formulary["Annual_Cost_Saving"] = (
    formulary["Original_Cost"]
    -
    formulary["Estimated_Alternative_Cost"]
)


formulary["Monthly_Cost_Saving"] = (
    formulary["Annual_Cost_Saving"]
    / 12
)


formulary["PMPM_Saving"] = np.nan


valid_pmpm = (
    formulary["Original_Benes"].notna()
    &
    (formulary["Original_Benes"] > 0)
)


formulary.loc[
    valid_pmpm,
    "PMPM_Saving"
] = (
    formulary.loc[
        valid_pmpm,
        "Monthly_Cost_Saving"
    ]
    /
    formulary.loc[
        valid_pmpm,
        "Original_Benes"
    ]
)


print("PMPM savings calculated.")

PMPM savings calculated.


In [107]:
# ============================================================
# FINAL FORMULARY RESULT
# ============================================================

result_columns = [
    "Trade_Name",
    "Alternative_Trade_Name",
    "TE_Code",
    "Alternative_TE_Code",
    "Original_Cost",
    "Estimated_Alternative_Cost",
    "Cost_Reduction_Pct",
    "Original_Benes",
    "Simulated_Access_Coverage_Pct",
    "PMPM_Saving",
    "Cost_Requirement",
    "Access_Requirement",
    "Final_Decision"
]


final_result = (
    formulary[result_columns]
    .sort_values(
        by="Cost_Reduction_Pct",
        ascending=False,
        na_position="last"
    )
    .copy()
)


display(
    final_result.head(30)
)

,Trade_Name,Alternative_Trade_Name,TE_Code,Alternative_TE_Code,Original_Cost,Estimated_Alternative_Cost,Cost_Reduction_Pct,Original_Benes,Simulated_Access_Coverage_Pct,PMPM_Saving,Cost_Requirement,Access_Requirement,Final_Decision
29,AZATHIOPRINE,AZASAN,AB,AB,1.253142e+07,1.065171e+07,15.0,73366.0,100.0,2.135087,True,True,RECOMMEND SWITCH
6,"GRISEOFULVIN, ULTRAMICROSIZE",FULVICIN P/G,AB,AB,2.345553e+05,1.993720e+05,15.0,6812.0,100.0,0.430408,True,True,RECOMMEND SWITCH
104,NORETHINDRONE,INCASSIA,AB1,AB1,2.588681e+05,2.200379e+05,15.0,12018.0,100.0,0.269250,True,True,RECOMMEND SWITCH
12,NORETHINDRONE,INCASSIA,AB2,AB1,2.588681e+05,2.200379e+05,15.0,12018.0,100.0,0.269250,True,True,RECOMMEND SWITCH
71,TIOPRONIN,VENXXIVA,AB,AB,2.712186e+07,2.305358e+07,15.0,5790.0,100.0,58.553230,True,True,RECOMMEND SWITCH
125,SAPROPTERIN DIHYDROCHLORIDE,ZELVYSIA,AB,AB,1.371863e+07,1.166083e+07,15.0,1357.0,100.0,126.369077,True,True,RECOMMEND SWITCH
2,WARFARIN SODIUM,JANTOVEN,AB,AB,4.024660e+07,3.420961e+07,15.0,857407.0,100.0,0.586749,True,True,RECOMMEND SWITCH
13,DICHLORPHENAMIDE,ORMALVI,AB,AB,9.536841e+06,8.106315e+06,15.0,3524.0,100.0,33.828182,True,True,RECOMMEND SWITCH
59,MIGLUSTAT,YARGESA,AB,AB,6.924978e+06,5.886232e+06,15.0,1028.0,100.0,84.204505,True,True,RECOMMEND SWITCH
69,DOFETILIDE,DOFETILDE,AB,AB,4.005866e+07,3.404986e+07,15.0,74984.0,100.0,6.677869,True,True,RECOMMEND SWITCH


In [108]:
# ============================================================
# SUMMARY
# ============================================================

total_te = len(formulary)

with_utilization = (
    formulary["Original_Cost"]
    .notna()
    .sum()
)

recommended = (
    formulary["Final_Decision"]
    == "RECOMMEND SWITCH"
).sum()

do_not_switch = (
    formulary["Final_Decision"]
    == "DO NOT SWITCH"
).sum()

data_needed = (
    formulary["Final_Decision"]
    == "DATA NEEDED"
).sum()


print("\n" + "=" * 65)
print("FORMULARY IMPACT ANALYSIS")
print("=" * 65)

print("Total TE alternatives:", total_te)
print("Drugs with utilization:", with_utilization)
print("Recommended switches:", recommended)
print("Do not switch:", do_not_switch)
print("Data needed:", data_needed)

print("=" * 65)


FORMULARY IMPACT ANALYSIS
Total TE alternatives: 132
Drugs with utilization: 41
Recommended switches: 41
Do not switch: 0
Data needed: 91


In [109]:
# ============================================================
# RECOMMENDED ALTERNATIVES
# ============================================================

recommended_alternatives = final_result[
    final_result["Final_Decision"]
    == "RECOMMEND SWITCH"
].copy()


print(
    "Recommended alternatives:",
    len(recommended_alternatives)
)


display(
    recommended_alternatives[
        [
            "Trade_Name",
            "Alternative_Trade_Name",
            "Cost_Reduction_Pct",
            "Simulated_Access_Coverage_Pct",
            "PMPM_Saving",
            "Final_Decision"
        ]
    ].head(30)
)

Recommended alternatives: 41


,Trade_Name,Alternative_Trade_Name,Cost_Reduction_Pct,Simulated_Access_Coverage_Pct,PMPM_Saving,Final_Decision
29,AZATHIOPRINE,AZASAN,15.0,100.0,2.135087,RECOMMEND SWITCH
6,"GRISEOFULVIN, ULTRAMICROSIZE",FULVICIN P/G,15.0,100.0,0.430408,RECOMMEND SWITCH
104,NORETHINDRONE,INCASSIA,15.0,100.0,0.269250,RECOMMEND SWITCH
12,NORETHINDRONE,INCASSIA,15.0,100.0,0.269250,RECOMMEND SWITCH
71,TIOPRONIN,VENXXIVA,15.0,100.0,58.553230,RECOMMEND SWITCH
125,SAPROPTERIN DIHYDROCHLORIDE,ZELVYSIA,15.0,100.0,126.369077,RECOMMEND SWITCH
2,WARFARIN SODIUM,JANTOVEN,15.0,100.0,0.586749,RECOMMEND SWITCH
13,DICHLORPHENAMIDE,ORMALVI,15.0,100.0,33.828182,RECOMMEND SWITCH
59,MIGLUSTAT,YARGESA,15.0,100.0,84.204505,RECOMMEND SWITCH
69,DOFETILIDE,DOFETILDE,15.0,100.0,6.677869,RECOMMEND SWITCH


In [110]:
# ============================================================
# SAVE FINAL OUTPUT
# ============================================================

output_dir = r"G:\PMS_Optimization\formulary_impact\outputs"

os.makedirs(
    output_dir,
    exist_ok=True
)


output_path = os.path.join(
    output_dir,
    "Final_Formulary_Impact_Analysis.csv"
)


final_result.to_csv(
    output_path,
    index=False
)


print("==============================================")
print("FORMULARY ANALYSIS SAVED SUCCESSFULLY")
print("==============================================")
print(output_path)

FORMULARY ANALYSIS SAVED SUCCESSFULLY
G:\PMS_Optimization\formulary_impact\outputs\Final_Formulary_Impact_Analysis.csv
